[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/06_Kernels/02_Softmax.ipynb)

# Softmax — Your First Reduction Kernel

This notebook builds on the [Vector Add kernel](./01_Vector_Add.ipynb) to introduce **reduction operations** — the first kernel where we compute *across* elements rather than independently per element.

Softmax is the perfect next kernel because:
- It appears in **every transformer** (attention mechanism)
- It's the simplest kernel where **fusion actually wins** against PyTorch
- It introduces `tl.max` and `tl.sum` — the reduction primitives you'll need for everything after

| | Vector Add (previous) | Softmax (this notebook) |
|---|---|---|
| Operation type | Element-wise | Row-wise reduction |
| Memory pattern | 1 read + 1 write per element | Read entire row, reduce, write row |
| PyTorch passes | 1 pass | 3 passes (max, exp-sub, sum-div) |
| Triton advantage | Minimal (memory-bound) | **Significant (fusion eliminates 2 extra passes)** |
| New Triton concepts | `tl.load`, `tl.store`, `mask` | `tl.max`, `tl.sum`, `tl.exp`, 2D indexing |

> We covered the **numerical stability** of softmax and the **online softmax algorithm** in the [Flash Attention notebook](../03_Training_Techniques/01_Flash_Attention.ipynb). This notebook focuses on implementing it as a fused GPU kernel.

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
# Detect runtime environment
import torch

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, '__IPYTHON__') else False
HAS_CUDA = torch.cuda.is_available()

if IN_COLAB:
    %pip install -q triton
    print(f"Running in Colab with GPU: {torch.cuda.get_device_name(0)}")
elif HAS_CUDA:
    print(f"Running locally with GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use Modal for remote GPU execution")
    print("Make sure you have Modal configured: pip install modal && modal token set")

## Why Softmax Is Different From Vector Add

Vector add processes each element independently — there's nothing to "fuse" because it's already one operation. Softmax requires **three dependent steps** across the entire row:

1. **Find the max** (for numerical stability)
2. **Subtract max and exponentiate** each element
3. **Sum the exponentials and divide** each element by the sum

PyTorch launches a separate CUDA kernel for each step, and each kernel reads the entire row from slow global memory (HBM):

```
PyTorch torch.softmax (3 passes over HBM):

  Pass 1: max         Pass 2: exp-sub        Pass 3: sum-div
  ┌─────────┐         ┌─────────┐            ┌─────────┐
  │ HBM     │ → max   │ HBM     │ → exp()    │ HBM     │ → / sum
  │ read x  │         │ read x  │ → write    │ read    │ → write
  └─────────┘         └─────────┘            └─────────┘
  3 reads + 2 writes from slow global memory


Triton fused softmax (1 pass):
  ┌─────────┐
  │ HBM     │ → SRAM: max → sub → exp → sum → div → write back
  │ read x  │   (all in fast on-chip memory)
  └─────────┘
  1 read + 1 write — that's it!
```

This is why kernel fusion matters — it's not about making the math faster, it's about **avoiding redundant trips to slow memory**.

## The Math

**Naive softmax:**

$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

**Problem:** If $\max(x) = 1000$, then $e^{1000} = \infty$. The computation overflows.

**Numerically stable softmax** (subtract the max first):

$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$

Now the largest exponent is $e^0 = 1$. The result is mathematically identical but won't overflow.

This is what both PyTorch and our Triton kernel implement.

## Naive Python Softmax (3-Pass)

This is conceptually what PyTorch does under the hood — three separate operations, each touching global memory:

In [2]:
import torch

def naive_softmax(x: torch.Tensor) -> torch.Tensor:
    """Softmax the PyTorch way — 3 separate passes over the data."""
    # Pass 1: Read x from HBM, compute row max, write max to HBM
    x_max = x.max(dim=-1, keepdim=True).values

    # Pass 2: Read x from HBM, subtract max, compute exp, write to HBM
    numerator = torch.exp(x - x_max)

    # Pass 3: Read numerator from HBM, compute sum, divide, write to HBM
    denominator = numerator.sum(dim=-1, keepdim=True)
    return numerator / denominator

## The Triton Kernel — Explained

Our Triton kernel fuses all three passes into one. The key differences from vector_add:

- **One program per row** — `tl.program_id(0)` gives the row index, not a block of elements
- **2D pointer arithmetic** — `row_idx * row_stride + col_offsets` to navigate a matrix
- **`BLOCK_SIZE` covers the entire row** — because we need the full row to compute max and sum
- **`-inf` padding** — out-of-bounds columns get `-inf` so `exp(-inf) = 0` and they vanish from the sum (using 0 would give `exp(0) = 1`, adding phantom probability mass)

The full code lives in `kernels/softmax.py`.

In [3]:
from kernels.softmax import softmax_kernel, softmax
import inspect
print(inspect.getsource(softmax_kernel))

ModuleNotFoundError: No module named 'triton'

## How the Grid Works

Contrast with vector_add, where the grid split a flat array into chunks:

```
Input matrix (n_rows x n_cols):
┌──────────────────────────────────────┐
│ Row 0: [x₀₀, x₀₁, ..., x₀d]       │ → Program 0 (full row softmax)
│ Row 1: [x₁₀, x₁₁, ..., x₁d]       │ → Program 1
│ Row 2: [x₂₀, x₂₁, ..., x₂d]       │ → Program 2
│  ...                                  │    ...
│ Row N: [xₙ₀, xₙ₁, ..., xₙd]       │ → Program N
└──────────────────────────────────────┘

Grid = (n_rows,)  ← one program per row
BLOCK_SIZE >= n_cols (each program loads the full row into SRAM)

Inside each program:
  1. Load entire row from HBM → SRAM
  2. tl.max → tl.exp(x - max) → tl.sum → divide
  3. Store result row from SRAM → HBM
```

| | Vector Add | Softmax |
|---|---|---|
| Grid | `(ceil(n / BLOCK_SIZE),)` | `(n_rows,)` |
| Each program handles | A chunk of independent elements | One full row |
| BLOCK_SIZE meaning | Elements per chunk | Must cover the entire row width |

In [4]:
# The Python wrapper handles BLOCK_SIZE and grid computation
print(inspect.getsource(softmax))

NameError: name 'inspect' is not defined

## Run on GPU

Triton requires an NVIDIA GPU. This notebook supports two execution modes:
- **Colab / Local CUDA** — runs directly on the available GPU
- **Modal** — runs on a remote T4 GPU (for Mac / no-GPU machines)

In [ ]:
import time

def benchmark_softmax():
    """Run correctness test + benchmark. Works on any CUDA device."""
    import triton
    import triton.language as tl

    @triton.jit
    def _softmax_kernel(
        input_ptr, output_ptr, n_rows, n_cols,
        input_row_stride, output_row_stride,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(axis=0)
        row_start_input = input_ptr + row_idx * input_row_stride
        row_start_output = output_ptr + row_idx * output_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols
        x = tl.load(row_start_input + col_offsets, mask=mask, other=-float('inf'))
        x_max = tl.max(x, axis=0)
        x = x - x_max
        numerator = tl.exp(x)
        denominator = tl.sum(numerator, axis=0)
        y = numerator / denominator
        tl.store(row_start_output + col_offsets, y, mask=mask)

    def triton_softmax(x):
        n_rows, n_cols = x.shape
        output = torch.empty_like(x)
        BLOCK_SIZE = triton.next_power_of_2(n_cols)
        grid = (n_rows,)
        _softmax_kernel[grid](
            x, output, n_rows, n_cols,
            x.stride(0), output.stride(0),
            BLOCK_SIZE=BLOCK_SIZE,
        )
        return output

    # --- Correctness test ---
    torch.manual_seed(0)
    x = torch.randn(128, 512, device="cuda")

    output_triton = triton_softmax(x)
    output_torch = torch.softmax(x, dim=-1)

    max_diff = (output_triton - output_torch).abs().max().item()
    match = torch.allclose(output_triton, output_torch, atol=1e-6)
    row_sums = output_triton.sum(dim=-1)
    print(f"Max difference: {max_diff:.2e}")
    print(f"Results match: {match}")
    print(f"Row sums — min: {row_sums.min():.6f}, max: {row_sums.max():.6f}")

    # --- Benchmark: vary n_cols with fixed n_rows=128 ---
    n_rows = 128
    col_sizes = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
    naive_times = []
    torch_times = []
    triton_times = []

    for n_cols in col_sizes:
        x = torch.randn(n_rows, n_cols, device="cuda")

        for _ in range(10):
            naive_softmax(x)
            torch.softmax(x, dim=-1)
            triton_softmax(x)
        torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(100):
            naive_softmax(x)
        torch.cuda.synchronize()
        naive_times.append((time.perf_counter() - start) / 100)

        start = time.perf_counter()
        for _ in range(100):
            torch.softmax(x, dim=-1)
        torch.cuda.synchronize()
        torch_times.append((time.perf_counter() - start) / 100)

        start = time.perf_counter()
        for _ in range(100):
            triton_softmax(x)
        torch.cuda.synchronize()
        triton_times.append((time.perf_counter() - start) / 100)

    return {
        "match": match,
        "max_diff": max_diff,
        "col_sizes": col_sizes,
        "naive_us": [t * 1e6 for t in naive_times],
        "torch_us": [t * 1e6 for t in torch_times],
        "triton_us": [t * 1e6 for t in triton_times],
    }

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_softmax()
else:
    # --- Modal remote execution (no local GPU) ---
    import modal

    app = modal.App("triton-softmax")
    image = modal.Image.debian_slim(python_version="3.11").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        return benchmark_softmax()

    with app.run():
        results = run_remote.remote()

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_softmax()
else:
    # --- Modal remote execution (no local GPU) ---
    # @triton.jit kernels can't be serialized by Modal, so all code is inline.
    import modal

    app = modal.App("triton-softmax")
    image = modal.Image.debian_slim(python_version="3.12").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        import torch, triton, triton.language as tl, time

        @triton.jit
        def _softmax_kernel(
            input_ptr, output_ptr, n_rows, n_cols,
            input_row_stride, output_row_stride, BLOCK_SIZE: tl.constexpr,
        ):
            row_idx = tl.program_id(axis=0)
            row_start = input_ptr + row_idx * input_row_stride
            col_offsets = tl.arange(0, BLOCK_SIZE)
            mask = col_offsets < n_cols
            x = tl.load(row_start + col_offsets, mask=mask, other=-float('inf'))
            x_max = tl.max(x, axis=0)
            numerator = tl.exp(x - x_max)
            denominator = tl.sum(numerator, axis=0)
            y = numerator / denominator
            out_start = output_ptr + row_idx * output_row_stride
            tl.store(out_start + col_offsets, y, mask=mask)

        def triton_softmax(x):
            n_rows, n_cols = x.shape
            output = torch.empty_like(x)
            BLOCK_SIZE = triton.next_power_of_2(n_cols)
            _softmax_kernel[(n_rows,)](
                x, output, n_rows, n_cols,
                x.stride(0), output.stride(0), BLOCK_SIZE=BLOCK_SIZE,
            )
            return output

        def naive_softmax(x):
            x_max = x.max(dim=-1, keepdim=True).values
            return torch.exp(x - x_max) / torch.exp(x - x_max).sum(dim=-1, keepdim=True)

        torch.manual_seed(0)
        x = torch.randn(128, 512, device="cuda")
        out_triton = triton_softmax(x)
        out_torch = torch.softmax(x, dim=-1)
        max_diff = (out_triton - out_torch).abs().max().item()
        match = torch.allclose(out_triton, out_torch, atol=1e-6)
        print(f"Max difference: {max_diff:.2e}, Results match: {match}")

        n_rows = 128
        col_sizes = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
        naive_times, torch_times, triton_times = [], [], []
        for n_cols in col_sizes:
            x = torch.randn(n_rows, n_cols, device="cuda")
            for _ in range(10):
                naive_softmax(x); torch.softmax(x, dim=-1); triton_softmax(x)
            torch.cuda.synchronize()
            for fn, times_list in [
                (lambda: naive_softmax(x), naive_times),
                (lambda: torch.softmax(x, dim=-1), torch_times),
                (lambda: triton_softmax(x), triton_times),
            ]:
                start = time.perf_counter()
                for _ in range(100): fn()
                torch.cuda.synchronize()
                times_list.append((time.perf_counter() - start) / 100)

        return {
            "match": match, "max_diff": max_diff, "col_sizes": col_sizes,
            "naive_us": [t * 1e6 for t in naive_times],
            "torch_us": [t * 1e6 for t in torch_times],
            "triton_us": [t * 1e6 for t in triton_times],
        }

    with app.run():
        results = run_remote.remote()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = results["col_sizes"]

# --- Plot 1: Execution time ---
ax = axes[0]
ax.plot(sizes, results["naive_us"], "^-.", label="Naive (3-pass)", linewidth=2, color="#e74c3c")
ax.plot(sizes, results["torch_us"], "s--", label="torch.softmax", linewidth=2, color="#3498db")
ax.plot(sizes, results["triton_us"], "o-", label="Triton (fused)", linewidth=2, color="#2ecc71")
ax.set_xscale("log", base=2)
ax.set_xlabel("Number of columns (row width)")
ax.set_ylabel("Time (microseconds)")
ax.set_title("Softmax: Execution Time vs Row Width")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Speedup ---
ax = axes[1]
speedup_vs_torch = [t / tr for t, tr in zip(results["torch_us"], results["triton_us"])]
speedup_vs_naive = [n / tr for n, tr in zip(results["naive_us"], results["triton_us"])]
x_pos = np.arange(len(sizes))
width = 0.35
ax.bar(x_pos - width/2, speedup_vs_torch, width, label="vs torch.softmax", color="#3498db")
ax.bar(x_pos + width/2, speedup_vs_naive, width, label="vs naive (3-pass)", color="#e74c3c")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel("Number of columns")
ax.set_ylabel("Speedup (higher = Triton wins)")
ax.set_title("Triton Speedup over PyTorch")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Why Triton Wins Here (But Not on Vector Add)

The key insight: **the speedup comes from fewer memory trips, not faster math.**

| Kernel | HBM Reads | HBM Writes | Total HBM Trips |
|---|---|---|---|
| PyTorch `torch.softmax` | 3 (max, exp-sub, div) | 2 (exp result, final) | **5** |
| Triton fused softmax | 1 (load row) | 1 (store row) | **2** |
| PyTorch `x + y` (vector add) | 1 | 1 | 2 |
| Triton vector add | 1 | 1 | 2 |

Vector add already does the minimum possible memory work (1 read + 1 write), so there's nothing to fuse. Softmax normally does 5 HBM trips — Triton cuts that to 2 by keeping the row in fast SRAM.

This principle extends to **Flash Attention**, which fuses the entire attention operation (matmul + softmax + matmul) to avoid materializing the $N \times N$ attention matrix in HBM.

## Connection to Attention

In every transformer's attention mechanism:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right) V$$

Softmax is applied **row-wise** to the $(N, N)$ score matrix, where $N$ is the sequence length. For sequence length $N = 4096$, that's 4096 rows, each of width 4096.

**Flash Attention** ([notebook](../03_Training_Techniques/01_Flash_Attention.ipynb)) takes the fusion idea from this notebook to its logical extreme: instead of just fusing softmax, it fuses the entire attention computation (matmul + softmax + matmul) into a single kernel using the **online softmax** algorithm to process the score matrix in tiles.

If you haven't read the Flash Attention notebook yet, this softmax kernel is the perfect warm-up — Flash Attention is essentially "fused softmax, but across tiles that don't fit in SRAM all at once."

## What to Notice

1. **Triton matches `torch.softmax` exactly** — the max difference should be near machine epsilon (~1e-7). Both use the same numerically stable algorithm.

2. **Triton beats PyTorch on this kernel** — unlike vector_add where PyTorch matched Triton, here fusion gives a measurable speedup because we eliminate 3 extra HBM round-trips.

3. **The naive 3-pass softmax is even slower** — it does even more memory traffic than `torch.softmax` (which has some internal optimizations). This confirms the speedup is about memory, not compute.

4. **`-inf` masking is critical** — padding with 0 instead of `-inf` would give `exp(0) = 1`, corrupting the denominator. This is the same issue that arises with **padding masks in attention** — masked positions must get `-inf` before softmax.

5. **Row width limit** — `BLOCK_SIZE` must be a power of 2 >= `n_cols`, and must fit in GPU shared memory (~16K float32 elements on T4). For very wide rows (>16K), you'd need a two-pass tiled approach. The [Triton tutorial](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html) covers this.

## Resources

- [Triton Fused Softmax Tutorial](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html) — Official Triton tutorial for this exact kernel
- [GPU MODE Lectures](https://github.com/gpu-mode/lectures) — Community GPU programming course